<a href="https://colab.research.google.com/github/AmyMugeni/cognilens/blob/Timetable_Extractor/Timetable_Exstaction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install docling

In [2]:
# Install the core OCR engine
!apt-get update && apt-get install -y tesseract-ocr libtesseract-dev poppler-utils

# Re-install Docling with full backend capabilities
!pip install --upgrade docling easyocr pytesseract



Hit:1 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Get:4 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Get:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.3 MB]
Hit:9 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Get:10 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 Packages [2,166 kB]
Get:11 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 Packages [1,702 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu noble/main amd64 Packages [3,052 kB]
Fetched 17.4 MB in 4s (4,287 kB/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources'

In [3]:
import os
from docling.datamodel.base_models import InputFormat
from docling.document_converter import DocumentConverter, PdfFormatOption
from google.colab import userdata

# Automatically loads your secret token securely into the environment
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    print("HF_TOKEN not found in Colab Secrets.")

In [5]:
import pdfplumber
import pandas as pd

pdf_path = "/content/ICS 4A (1).pdf"
csv_output_path = "/content/final_timetable.csv"

# Define the verified grid boundaries from coordinate extraction
time_slots = [
    "8.15 - 9.15", "9.15 – 10.15", "10.15 – 11.15", "11.15 – 12.15",
    "12.15 - 1.15", "1.15 - 2.15", "2.15 – 3.15", "3.15 – 4.15", "4.15 - 5.15"
]

# Map X coordinate intervals to column indices
x_bounds = [99.0, 165.1, 224.1, 298.8, 377.3, 448.0, 514.9, 586.6, 664.1, 741.9]

def get_col_idx(x_mid):
    for i in range(len(x_bounds) - 1):
        if x_bounds[i] <= x_mid < x_bounds[i+1]:
            return i
    return -1

# Map Y coordinate intervals to days of the week
def get_day_of_week(y):
    if 80 <= y < 150: return "Monday"
    if 150 <= y < 220: return "Tuesday"
    if 220 <= y < 296: return "Wednesday"
    if 296 <= y < 373: return "Thursday"
    if 373 <= y < 460: return "Friday"
    return None

# Parse document words
with pdfplumber.open(pdf_path) as pdf:
    page = pdf.pages[0]
    words = page.extract_words(x_tolerance=3, y_tolerance=3)

# Initialize schedule structure
schedule = {day: {slot: [] for slot in time_slots} for day in ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]}

# Populate words into their exact logical grid cells
for w in words:
    day = get_day_of_week(w['top'])
    if not day:
        continue

    x_mid = (w['x0'] + w['x1']) / 2
    col_idx = get_col_idx(x_mid)
    if col_idx == -1:
        continue

    slot = time_slots[col_idx]
    schedule[day][slot].append(w)

# Format words into clean text within each cell
final_rows = []
for day in ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]:
    row_data = {"Day": day}
    for slot in time_slots:
        cell_words = schedule[day][slot]
        if not cell_words:
            row_data[slot] = "-"
            continue

        # Sort words in cell top-to-bottom, then left-to-right
        sorted_words = sorted(cell_words, key=lambda x: (round(x['top'] / 3) * 3, x['x0']))

        # Assemble text
        lines = []
        curr_line = []
        last_top = None
        for w in sorted_words:
            if last_top is None or abs(w['top'] - last_top) < 4:
                curr_line.append(w['text'])
            else:
                lines.append(" ".join(curr_line))
                curr_line = [w['text']]
            last_top = w['top']
        if curr_line:
            lines.append(" ".join(curr_line))

        # Standardize space separation inside CSV output
        row_data[slot] = " | ".join(lines)
    final_rows.append(row_data)

# Export and Display DataFrame
df_final = pd.DataFrame(final_rows)
df_final.to_csv(csv_output_path, index=False)

print("Saved clean timetable grid to:", csv_output_path)
display(df_final)

Saved clean timetable grid to: /content/final_timetable.csv


,Day,8.15 - 9.15,9.15 – 10.15,10.15 – 11.15,11.15 – 12.15,12.15 - 1.15,1.15 - 2.15,2.15 – 3.15,3.15 – 4.15,4.15 - 5.15
0,Monday,Computer,ICS 4106 | Simulation and | STM-B F2-05,Modelling,ICS 4102 | Machine Learning | Meeting ID: | 36...,-,ICS 4103 | Cloud Computing | Meeting ID: | 388...,427,ICS | Multimedia | Meeting | 313 942,3105 | Applications | ID: | 404 273 032 | Pass...
1,Tuesday,363,C Ouma | ICS 4101 | ICS Project II | Meeting I...,Maisha | Gents,J Wekesa | Program 7 | Maisha | 4A. RM B,Program,9vx9sm69 | V Ochango | Distributed,ICS 4104 | Systems | STM-B F2-05,Z,Muti
2,Wednesday,Embedded,Passcode: | rk6is9cp | K Omondi | ICS 4111 | S...,P | of Things,7 Ladies | ICS 4A | 10 | Neri R Bunde | ICS | ...,RM | 4106 | and,Distributed,H Talo | ICS 4104 | Systems,ICS,4103
3,Thursday,315,378 662 024 967 | Passcode: | R5XH9Pd9 | A Vikiru,-,Meeting | 378 219 | 79 | Passcode: | Dv7JR7zZ ...,ID: | 356 369,320,Meeting ID: | 379 169 459 244 | Passcode: | tb...,Cloud | V,Computing | MSB 13 | Ochango
4,Friday,-,ICS 4101 | ICS Project II | LT 6 | K Omondi | ...,-,Embedded,ICS 4111 | Systems and Internet | of Things | ...,-,-,ICS | Multimedia | Z,3105 | Applications | LT 6 | Muti
